# Python Basics Revision for Bloomberg Interview

This notebook is a focused refresh for a **first-round Bloomberg interview** for a **Senior Machine Learning Engineer** role. It emphasizes Python fundamentals that are often used to probe coding fluency, debugging judgment, and practical engineering taste.

## How to use this notebook
- Read the short summary in each section.
- Try the exercises before looking at the sample solutions.
- Prefer writing clean, explainable code over clever one-liners.
- When relevant, state time and space complexity out loud.

## Interview Priorities

For a senior ML engineering screen, expect Python questions around:

1. Core syntax and semantics: mutability, scope, default arguments, truthiness.
2. Data structures: lists, tuples, dicts, sets, heaps, counters.
3. Functions: `*args`, `**kwargs`, closures, decorators, generators.
4. Object model: classes, dataclasses, magic methods, inheritance tradeoffs.
5. Reliability: exceptions, context managers, logging, typing.
6. Practical coding: parsing data, transforming collections, writing maintainable code quickly.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from functools import wraps
from heapq import heappush, heappop
from typing import Iterable

print('Notebook ready')

Notebook ready


## 1. Mutability, Assignment, and Default Arguments

### Key points
- Assignment binds a name to an object; it does not copy the object.
- Lists and dictionaries are mutable; tuples, strings, and integers are immutable.
- Avoid mutable default arguments because the same object is reused across calls.

In [2]:
def append_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

def append_item_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print('bad #1 ->', append_item_bad(1))
print('bad #2 ->', append_item_bad(2))
print('good #1 ->', append_item_good(1))
print('good #2 ->', append_item_good(2))

bad #1 -> [1]
bad #2 -> [1, 2]
good #1 -> [1]
good #2 -> [2]


### Exercise
Write a function `safe_increment(counter, key)` that increments a dictionary counter and returns the updated dictionary. It should create a new dictionary when `counter` is not provided.

In [3]:
def safe_increment(counter=None, key='event'):
    # Write your answer here
    pass

In [ ]:
def safe_increment_solution(counter=None, key='event'):
    if counter is None:
        counter = {}
    counter[key] = counter.get(key, 0) + 1
    return counter

print(safe_increment_solution())
print(safe_increment_solution({'event': 2}, 'event'))

## 2. Lists, Dicts, Sets, and Common Patterns

### What to remember
- Use lists for ordered sequences, sets for fast membership checks, and dicts for keyed lookups.
- Dictionary and set lookup are average-case $O(1)$.
- Prefer comprehensions when they improve clarity.

In [4]:
prices = [101, 103, 99, 105, 103, 101]
unique_prices = set(prices)
price_counts = Counter(prices)
price_to_index = {value: index for index, value in enumerate(prices)}

print('unique:', unique_prices)
print('counts:', price_counts)
print('last index map:', price_to_index)

unique: {105, 99, 101, 103}
counts: Counter({101: 2, 103: 2, 99: 1, 105: 1})
last index map: {101: 5, 103: 4, 99: 2, 105: 3}


### Exercise
Given a list of words, return the first non-repeating word while preserving order.

In [5]:
def first_non_repeating(words):
    # Write your answer here
    pass

sample_words = ['ml', 'python', 'ml', 'bloomberg', 'data', 'data']

In [ ]:
def first_non_repeating_solution(words):
    counts = Counter(words)
    for word in words:
        if counts[word] == 1:
            return word
    return None

print(first_non_repeating_solution(sample_words))

## 3. Functions, Closures, and Decorators

Senior candidates are often expected to explain:
- lexical scoping
- closures
- when decorators are useful
- how to preserve metadata with `functools.wraps`

In [6]:
def timing_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f'called {func.__name__} with args={args}, kwargs={kwargs}')
        return result
    return wrapper

@timing_decorator
def normalize(values):
    total = sum(values)
    return [value / total for value in values]

print(normalize([1, 2, 3]))

called normalize with args=([1, 2, 3],), kwargs={}
[0.16666666666666666, 0.3333333333333333, 0.5]


In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return factor * value
    return multiply

triple = make_multiplier(3)
print(triple(5))

## 4. Iterators, Generators, and Memory Efficiency

Good interview framing:
- Iterables can be looped over.
- Iterators produce items one by one via `__next__`.
- Generators are a concise way to build iterators and help with memory efficiency.

In [ ]:
def rolling_average(values: Iterable[int], window_size: int):
    window = deque(maxlen=window_size)
    for value in values:
        window.append(value)
        if len(window) == window_size:
            yield sum(window) / window_size

print(list(rolling_average([1, 2, 3, 4, 5], 3)))

### Exercise
Implement a generator `running_max(values)` that yields the maximum seen so far at each step.

In [ ]:
def running_max(values):
    # Write your answer here
    pass

In [ ]:
def running_max_solution(values):
    current = None
    for value in values:
        current = value if current is None else max(current, value)
        yield current

print(list(running_max_solution([2, 1, 5, 3, 6])))

## 5. Caching and Memoization

Caching is a common first-round interview topic because it tests whether you can trade memory for speed and recognize repeated work.

### What to know

- `Memoization` stores results of function calls so repeated calls with the same inputs are fast.
- `LRU cache` keeps only the most recently used results and evicts older ones when capacity is limited.
- This is useful when the same expensive subproblem appears many times, such as recursive dynamic programming, repeated API results, or reused numerical calculations.

### Interview patterns

#### 1. Fibonacci with memoization
- Naive recursive Fibonacci repeats the same subproblems many times.
- Caching turns the exponential recursion tree into roughly linear work.
- This is the classic first-round example for explaining memoization.

#### 2. Cached numerical integration
- If you repeatedly integrate the same function over the same interval, cache the result by function name and parameters.
- This is a practical way to discuss expensive pricing or risk calculations.

#### 3. LRU-style thinking
- If memory is limited, you cannot keep everything forever.
- LRU is a standard eviction strategy because it is simple and often effective in practice.

### What interviewers want to hear

- what repeated work is being eliminated
- the tradeoff between time and memory
- when caching is unsafe because inputs are mutable or side effects matter
- when cache invalidation becomes a real design problem

In [8]:
import numpy as np

fib_cache = {0: 0, 1: 1}

def fibonacci_cached(n: int) -> int:
    if n in fib_cache:
        return fib_cache[n]

    fib_cache[n] = fibonacci_cached(n - 1) + fibonacci_cached(n - 2)
    return fib_cache[n]


def integrate_trapezoid_cached():
    cache: dict[tuple[str, float, float, int], float] = {}

    def integrate(function_name: str, func, left: float, right: float, steps: int = 1000) -> float:
        key = (function_name, left, right, steps)
        if key in cache:
            return cache[key]

        grid = np.linspace(left, right, steps + 1)
        values = func(grid)
        area = np.trapezoid(values, grid)
        cache[key] = area
        return area

    return integrate


integrate = integrate_trapezoid_cached()
fibonacci_values = [fibonacci_cached(n) for n in range(10)]
integral_1 = integrate('x_squared', lambda x: x**2, 0.0, 1.0, steps=500)
integral_2 = integrate('x_squared', lambda x: x**2, 0.0, 1.0, steps=500)

print('fibonacci:', fibonacci_values)
print('integral first call:', integral_1)
print('integral second call from cache:', integral_2)
print('fib cache keys:', sorted(fib_cache))

fibonacci: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
integral first call: 0.333334
integral second call from cache: 0.333334
fib cache keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 6. Sorting, Heap, and Top-K

Bloomberg-style screens often include practical collection tasks such as top-$k$, grouping, sorting with keys, or streaming updates.

In [ ]:
records = [
    {'symbol': 'AAPL', 'volume': 120},
    {'symbol': 'MSFT', 'volume': 95},
    {'symbol': 'NVDA', 'volume': 180},
]

sorted_records = sorted(records, key=lambda item: item['volume'], reverse=True)
print(sorted_records)

heap = []
for value in [5, 1, 7, 3]:
    heappush(heap, value)

print([heappop(heap) for _ in range(len(heap))])

## 6A. Mostly Asked Sorting Algorithms

These are the sorting algorithms interviewers most often expect you to recognize and explain quickly.

### Bubble Sort
- Repeatedly compare adjacent elements and swap them if they are out of order.
- After each pass, the largest remaining element "bubbles" to the end.
- Time complexity: worst and average $O(n^2)$, best $O(n)$ with an early-exit optimization.
- Space complexity: $O(1)$.
- Good for explaining basic sorting mechanics, but rarely used in production.

### Merge Sort
- Split the array into halves until each piece has one element.
- Merge the pieces back together in sorted order.
- Time complexity: $O(n \log n)$ in all cases.
- Space complexity: $O(n)$ for the merge buffer.
- Good when you want predictable performance and a clean divide-and-conquer story.

### Quick Sort
- Pick a pivot, partition the array into elements smaller and larger than the pivot, then recurse.
- Average time complexity: $O(n \log n)$.
- Worst-case time complexity: $O(n^2)$ when partitions are very unbalanced.
- Space complexity: typically $O(\log n)$ recursion depth.
- Often preferred in interviews because it shows partitioning, recursion, and tradeoff awareness.

### Heap Sort
- Build a heap, then repeatedly extract the smallest or largest element.
- Time complexity: $O(n \log n)$.
- Space complexity: $O(1)$ for an in-place variant, though Python examples often use `heapq` with extra storage.
- Useful when discussing top-$k$, streaming problems, and priority queues.

### Python Reality Check
- In real Python code, prefer `sorted()` or `list.sort()`, which use Timsort.
- Timsort is stable and performs very well on partially sorted real-world data.
- In interviews, the point is usually to explain *why* one algorithm is a better fit than another.

In [2]:
def bubble_sort(values):
    items = values[:]
    n = len(items)

    for end in range(n - 1, 0, -1):
        swapped = False
        for index in range(end):
            if items[index] > items[index + 1]:
                items[index], items[index + 1] = items[index + 1], items[index]
                swapped = True
        if not swapped:
            break

    return items


def merge_sort(values):
    if len(values) <= 1:
        return values[:]

    midpoint = len(values) // 2
    left = merge_sort(values[:midpoint])
    right = merge_sort(values[midpoint:])

    merged = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def quick_sort(values):
    if len(values) <= 1:
        return values[:]

    pivot = values[len(values) // 2]
    lower = [value for value in values if value < pivot]
    equal = [value for value in values if value == pivot]
    higher = [value for value in values if value > pivot]
    return quick_sort(lower) + equal + quick_sort(higher)


def heap_sort(values):
    heap = []
    for value in values:
        heappush(heap, value)
    return [heappop(heap) for _ in range(len(heap))]


sample = [7, 2, 9, 1, 5, 3]
print('bubble ->', bubble_sort(sample))
print('merge  ->', merge_sort(sample))
print('quick  ->', quick_sort(sample))
print('heap   ->', heap_sort(sample))

bubble -> [1, 2, 3, 5, 7, 9]
merge  -> [1, 2, 3, 5, 7, 9]
quick  -> [1, 2, 3, 5, 7, 9]
heap   -> [1, 2, 3, 5, 7, 9]


### What to say in an interview

- If asked to implement a sorting algorithm from scratch, `merge sort` is usually the safest clean answer.
- If asked about tradeoffs, compare `merge sort`, `quick sort`, and Python’s built-in Timsort.
- If the problem is really top-$k$ rather than full sorting, mention a heap instead of sorting everything.
- If stability matters, explicitly say so and mention that merge sort and Timsort are stable.

## 7. Classes, Dataclasses, and Special Methods

Know when a lightweight class clarifies intent. For interview code, `@dataclass` is often a good default for simple state containers.

In [ ]:
@dataclass(order=True)
class ModelScore:
    score: float
    name: str

best = max([
    ModelScore(0.81, 'baseline'),
    ModelScore(0.86, 'xgboost'),
    ModelScore(0.84, 'lightgbm'),
])

print(best)

### Prompt to answer verbally
What is the difference between `__repr__` and `__str__`, and when would you implement each?

## 8. Exceptions, Context Managers, and Defensive Coding

Interviewers often care less about the exact exception type than about whether your code fails clearly and safely.

In [ ]:
def parse_ratio(text: str) -> float:
    numerator_text, denominator_text = text.split('/')
    numerator = float(numerator_text)
    denominator = float(denominator_text)
    if denominator == 0:
        raise ValueError('denominator must be non-zero')
    return numerator / denominator

for sample in ['3/2', '10/5']:
    print(sample, '->', parse_ratio(sample))

## 9. Typing and Clean Interfaces

For a senior role, type hints signal clarity. They do not replace tests, but they do make interfaces easier to reason about.

In [ ]:
def group_by_length(words: list[str]) -> dict[int, list[str]]:
    grouped: dict[int, list[str]] = defaultdict(list)
    for word in words:
        grouped[len(word)].append(word)
    return dict(grouped)

print(group_by_length(['alpha', 'ml', 'data', 'ai']))

## 10. Pandas Operations for Quant Finance

These are the pandas operations you are most likely to use when working with market, portfolio, and factor data.

### High-value operations to revise

- `read_csv(..., parse_dates=[...])`: load time series data with proper datetime parsing from the start.
- `set_index`, `sort_index`: make time-based operations predictable and correct.
- column arithmetic: compute returns, spreads, PnL, exposure, and z-scores.
- `shift`: build lagged features and previous-day comparisons without look-ahead bias.
- `pct_change`: compute simple returns quickly for prices or index levels.
- `rolling`: calculate rolling mean, volatility, drawdown helpers, and moving statistics.
- `resample`: aggregate intraday data to daily, weekly, or monthly frequency.
- `groupby`: compute per-symbol or per-bucket statistics across a panel dataset.
- `merge` and `join`: combine prices, positions, benchmark data, and risk factors.
- `fillna`, `ffill`, `dropna`: handle missing observations explicitly.

### Quant finance framing

If you are discussing pandas in an interview, tie the operation to a concrete use case:
- returns from price series
- rolling volatility from returns
- grouped metrics by symbol or sector
- lagged features for forecasting or signal generation
- resampling from tick or daily data into a strategy horizon

In [3]:
import pandas as pd

prices = pd.DataFrame(
    {
        'date': pd.date_range('2024-01-01', periods=6, freq='D'),
        'symbol': ['AAPL', 'AAPL', 'AAPL', 'MSFT', 'MSFT', 'MSFT'],
        'close': [100.0, 102.0, 101.0, 200.0, 204.0, 202.0],
        'volume': [1000, 1100, 1050, 2000, 2100, 1900],
    }
)

prices = prices.sort_values(['symbol', 'date'])
prices['return'] = prices.groupby('symbol')['close'].pct_change()
prices['prev_close'] = prices.groupby('symbol')['close'].shift(1)
prices['rolling_vol_2d'] = (
    prices.groupby('symbol')['return']
    .rolling(window=2)
    .std()
    .reset_index(level=0, drop=True)
)

print('panel data')
print(prices)

daily_summary = (
    prices.groupby('symbol')
    .agg(
        avg_volume=('volume', 'mean'),
        last_close=('close', 'last'),
        total_return=('close', lambda series: series.iloc[-1] / series.iloc[0] - 1),
    )
)

print('\nper-symbol summary')
print(daily_summary)

wide_prices = prices.pivot(index='date', columns='symbol', values='close')
print('\nwide price table')
print(wide_prices)

resampled = (
    wide_prices
    .resample('2D')
    .last()
    .ffill()
)

print('\nresampled prices')
print(resampled)

panel data
        date symbol  close  volume    return  prev_close  rolling_vol_2d
0 2024-01-01   AAPL  100.0    1000       NaN         NaN             NaN
1 2024-01-02   AAPL  102.0    1100  0.020000       100.0             NaN
2 2024-01-03   AAPL  101.0    1050 -0.009804       102.0        0.021075
3 2024-01-04   MSFT  200.0    2000       NaN         NaN             NaN
4 2024-01-05   MSFT  204.0    2100  0.020000       200.0             NaN
5 2024-01-06   MSFT  202.0    1900 -0.009804       204.0        0.021075

per-symbol summary
        avg_volume  last_close  total_return
symbol                                      
AAPL        1050.0       101.0          0.01
MSFT        2000.0       202.0          0.01

wide price table
symbol       AAPL   MSFT
date                    
2024-01-01  100.0    NaN
2024-01-02  102.0    NaN
2024-01-03  101.0    NaN
2024-01-04    NaN  200.0
2024-01-05    NaN  204.0
2024-01-06    NaN  202.0

resampled prices
symbol       AAPL   MSFT
date             

### Advanced pandas `where` and `mask`

`where` is useful when you want to keep values that satisfy a condition and replace the rest. `mask` is the inverse: it replaces values where the condition is true.

### Why it matters in finance

This pattern is common when you want to:
- cap extreme returns without dropping rows
- keep only long signals or only short signals
- suppress invalid prices or volumes
- build conditional exposures without writing loops

### Mental model

- `series.where(condition, other)` keeps the original value where `condition` is true.
- where `condition` is false, it uses `other`.
- `mask(condition, other)` does the opposite.
- This is often cleaner than nested `np.where` when you are already working in pandas with aligned indexes.

In [11]:
where_frame = prices.copy()
where_frame['capped_return'] = where_frame['return'].where(
    where_frame['return'].abs() <= 0.015,
    other=0.015 * where_frame['return'].pipe(lambda series: series / series.abs()),
)
where_frame['long_only_signal'] = where_frame['return'].where(where_frame['return'] > 0, other=0.0)
where_frame['volume_status'] = pd.Series('ok', index=where_frame.index).mask(
    where_frame['volume'] < 1050,
    other='low_volume',
)

print(where_frame[['symbol', 'date', 'return', 'capped_return', 'long_only_signal', 'volume_status']])

  symbol       date    return  capped_return  long_only_signal volume_status
0   AAPL 2024-01-01       NaN            NaN              0.00    low_volume
1   AAPL 2024-01-02  0.020000       0.015000              0.02            ok
2   AAPL 2024-01-03 -0.009804      -0.009804              0.00            ok
3   MSFT 2024-01-04       NaN            NaN              0.00            ok
4   MSFT 2024-01-05  0.020000       0.015000              0.02            ok
5   MSFT 2024-01-06 -0.009804      -0.009804              0.00            ok


## 11. NumPy for Finance

NumPy is the right tool when you want fast vectorized numerical operations on arrays, returns, covariance matrices, and simulations.

### Operations worth revising

- `np.array`, broadcasting, and elementwise arithmetic for returns and exposures.
- `np.diff` and slicing for price changes and lagged comparisons.
- `np.log`, `np.exp`, and `np.cumsum` for log returns and cumulative performance.
- `np.mean`, `np.std`, `np.percentile` for summary statistics and risk metrics.
- `np.dot` and `@` for portfolio return aggregation and linear algebra.
- `np.cov` and `np.corrcoef` for covariance and correlation analysis.
- boolean masks with `np.where` for signal logic and threshold rules.
- `np.maximum.accumulate` for drawdown calculations.
- random sampling for Monte Carlo style scenarios.

### Finance framing

In interviews, connect NumPy to concrete tasks:
- vectorized return calculations across many assets
- portfolio weights multiplied by asset returns
- covariance-based risk estimation
- drawdown and volatility calculations
- Monte Carlo path generation for prices or PnL

In [4]:
import numpy as np

close_prices = np.array([100.0, 102.0, 101.0, 105.0, 107.0])
simple_returns = close_prices[1:] / close_prices[:-1] - 1
log_returns = np.diff(np.log(close_prices))

weights = np.array([0.5, 0.3, 0.2])
asset_returns = np.array([0.01, -0.005, 0.012])
portfolio_return = weights @ asset_returns

return_matrix = np.array(
    [
        [0.010, 0.004, -0.002],
        [0.005, -0.003, 0.007],
        [-0.004, 0.006, 0.003],
        [0.012, 0.002, -0.001],
    ]
)
covariance = np.cov(return_matrix, rowvar=False)
correlation = np.corrcoef(return_matrix, rowvar=False)

equity_curve = np.array([100, 103, 101, 106, 104, 110], dtype=float)
running_peak = np.maximum.accumulate(equity_curve)
drawdown = equity_curve / running_peak - 1

rng = np.random.default_rng(42)
shock_scenarios = rng.normal(loc=0.0005, scale=0.01, size=5)

print('simple returns:', simple_returns)
print('log returns:', log_returns)
print('portfolio return:', portfolio_return)
print('\ncovariance matrix:\n', covariance)
print('\ncorrelation matrix:\n', correlation)
print('\ndrawdown:', drawdown)
print('\nshock scenarios:', shock_scenarios)

simple returns: [ 0.02       -0.00980392  0.03960396  0.01904762]
log returns: [ 0.01980263 -0.0098523   0.03883983  0.01886848]
portfolio return: 0.005900000000000001

covariance matrix:
 [[ 5.09166667e-05 -8.91666667e-06 -1.64166667e-05]
 [-8.91666667e-06  1.49166667e-05 -9.58333333e-06]
 [-1.64166667e-05 -9.58333333e-06  1.69166667e-05]]

correlation matrix:
 [[ 1.         -0.32354646 -0.55936798]
 [-0.32354646  1.         -0.60328608]
 [-0.55936798 -0.60328608  1.        ]]

drawdown: [ 0.          0.         -0.01941748  0.         -0.01886792  0.        ]

shock scenarios: [ 0.00354717 -0.00989984  0.00800451  0.00990565 -0.01901035]


## 12. Grouping With NumPy

NumPy does not have a direct `groupby` method like pandas, but first-round interviews sometimes ask how you would aggregate by group using only arrays.

### What to know

- Use `np.unique(..., return_inverse=True)` to map labels to integer group ids.
- Use `np.bincount` for fast grouped counts and grouped sums.
- Use `np.add.at` when you want explicit indexed accumulation.
- This is useful when you want lower-level array control or need to show you understand how grouping works under the hood.

### Common interview pattern

Given arrays of symbols and values, compute grouped sums, counts, or means without pandas.

### Finance framing

This comes up naturally in finance when you want to aggregate:
- PnL by desk
- exposure by sector
- volume by symbol
- factor contribution by bucket

In [9]:
symbols = np.array(['Tech', 'Tech', 'Banks', 'Energy', 'Banks', 'Tech'])
pnl = np.array([1.2, -0.3, 0.8, -0.1, 0.4, 0.9])

groups, inverse = np.unique(symbols, return_inverse=True)
group_counts = np.bincount(inverse)
group_sums = np.bincount(inverse, weights=pnl)
group_means = group_sums / group_counts

accumulated = np.zeros(len(groups))
np.add.at(accumulated, inverse, pnl)

print('groups:', groups)
print('counts:', group_counts)
print('sums via bincount:', group_sums)
print('means:', group_means)
print('sums via add.at:', accumulated)

groups: ['Banks' 'Energy' 'Tech']
counts: [2 1 3]
sums via bincount: [ 1.2 -0.1  1.8]
means: [ 0.6 -0.1  0.6]
sums via add.at: [ 1.2 -0.1  1.8]


## 13. Most Asked Financial Indicators

These are the indicators interviewers most often expect you to recognize, explain, and compute from a price series.

### Indicators worth knowing

#### 1. Simple Moving Average (SMA)
- Definition: the rolling arithmetic mean of the last $n$ prices.
- What it tells you: it smooths noisy price action and highlights the underlying trend.
- How to interpret it: when price is above the SMA, traders often say the short-term trend is stronger; when price is below it, momentum may be weaker.
- Limitation: it reacts slowly because every observation in the window gets equal weight.

#### 2. Exponential Moving Average (EMA)
- Definition: a weighted moving average that puts more emphasis on recent prices.
- What it tells you: similar to SMA, but it reacts faster to recent market moves.
- How to interpret it: useful when you want a trend signal that adapts more quickly than a simple rolling mean.
- Limitation: it can be more sensitive to short-term noise.

#### 3. Rolling Volatility
- Definition: the rolling standard deviation of returns over a chosen window.
- What it tells you: how variable or risky returns have been recently.
- How to interpret it: a higher value means more dispersion in returns and usually more uncertainty or risk.
- Limitation: volatility measures magnitude of movement, not direction, so it does not tell you whether the market is going up or down.

#### 4. Relative Strength Index (RSI)
- Definition: a momentum oscillator based on the ratio of recent average gains to recent average losses.
- What it tells you: whether recent price changes have been mostly upward or mostly downward.
- How to interpret it: high RSI is often described as overbought and low RSI as oversold, but that is context-dependent and not a standalone trading rule.
- Limitation: in strong trends, RSI can stay high or low for a long time, so it should not be treated as a guaranteed reversal signal.

#### 5. Bollinger Bands
- Definition: a moving average with an upper and lower band set a chosen number of standard deviations away.
- What it tells you: both the local trend center and how wide recent price variation has been.
- How to interpret it: wider bands suggest higher recent volatility; narrower bands suggest calmer conditions.
- Limitation: touching a band does not automatically mean reversal; it may simply reflect strong trend continuation.

#### 6. MACD
- Definition: the difference between a fast EMA and a slow EMA, often paired with an EMA of that difference called the signal line.
- What it tells you: whether short-term momentum is accelerating or weakening relative to the longer trend.
- How to interpret it: crossovers between MACD and the signal line are often used as momentum-change signals.
- Limitation: like most smoothed indicators, it can lag and produce false signals in choppy markets.

#### 7. Drawdown
- Definition: the percentage drop from the running peak of a price series, equity curve, or portfolio value.
- What it tells you: how painful the losses have been from the investor's point of view.
- How to interpret it: maximum drawdown is one of the most common downside risk statistics in finance and backtesting.
- Limitation: it depends on the historical path, so it is descriptive of realized stress rather than a full forecast of future risk.

### How to explain them in an interview

When asked about indicators, do not just name them. Explain:
- what input series they use, usually prices or returns
- whether they are trend, momentum, or risk indicators
- what window or smoothing choice changes their behavior
- where look-ahead bias can accidentally creep in during backtests

### Common interview follow-ups

- Why might EMA be preferred over SMA in a fast-moving market?
- Why can RSI give misleading signals in a strong trend?
- Why is drawdown often more intuitive to portfolio managers than variance?
- Why should indicators be validated out of sample instead of trusted directly?

In [5]:
indicator_prices = pd.Series(
    [100.0, 101.0, 103.0, 102.0, 104.0, 107.0, 106.0, 108.0],
    index=pd.date_range('2024-02-01', periods=8, freq='D'),
    name='close',
)

indicator_returns = indicator_prices.pct_change()
sma_3 = indicator_prices.rolling(window=3).mean()
ema_3 = indicator_prices.ewm(span=3, adjust=False).mean()
rolling_vol_3 = indicator_returns.rolling(window=3).std()

delta = indicator_prices.diff()
gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)
avg_gain = gains.rolling(window=3).mean()
avg_loss = losses.rolling(window=3).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
rsi_3 = 100 - (100 / (1 + rs))

middle_band = sma_3
band_std = indicator_prices.rolling(window=3).std()
upper_band = middle_band + 2 * band_std
lower_band = middle_band - 2 * band_std

ema_fast = indicator_prices.ewm(span=3, adjust=False).mean()
ema_slow = indicator_prices.ewm(span=5, adjust=False).mean()
macd = ema_fast - ema_slow
signal_line = macd.ewm(span=3, adjust=False).mean()

indicator_drawdown = indicator_prices / indicator_prices.cummax() - 1

indicator_frame = pd.DataFrame(
    {
        'close': indicator_prices,
        'sma_3': sma_3,
        'ema_3': ema_3,
        'rolling_vol_3': rolling_vol_3,
        'rsi_3': rsi_3,
        'upper_band': upper_band,
        'lower_band': lower_band,
        'macd': macd,
        'signal_line': signal_line,
        'drawdown': indicator_drawdown,
    }
)

print(indicator_frame.round(4))

            close     sma_3     ema_3  rolling_vol_3    rsi_3  upper_band  \
2024-02-01  100.0       NaN  100.0000            NaN      NaN         NaN   
2024-02-02  101.0       NaN  100.5000            NaN      NaN         NaN   
2024-02-03  103.0  101.3333  101.7500            NaN      NaN    104.3884   
2024-02-04  102.0  102.0000  101.8750         0.0150  75.0000    104.0000   
2024-02-05  104.0  103.0000  102.9375         0.0170  80.0000    105.0000   
2024-02-06  107.0  104.3333  104.9688         0.0201  83.3333    109.3666   
2024-02-07  106.0  105.6667  105.4844         0.0199  83.3333    108.7217   
2024-02-08  108.0  107.0000  106.7422         0.0198  83.3333    109.0000   

            lower_band    macd  signal_line  drawdown  
2024-02-01         NaN  0.0000       0.0000    0.0000  
2024-02-02         NaN  0.1667       0.0833    0.0000  
2024-02-03     98.2783  0.5278       0.3056    0.0000  
2024-02-04    100.0000  0.3935       0.3495   -0.0097  
2024-02-05    101.0000  0.

## 14. Practical Mini Problems

Try these in 5 to 10 minutes each.

1. Merge two sorted lists without using `sorted()`.
2. Return the top 2 most common words from a document.
3. Given trades as dictionaries, group them by symbol and compute total quantity.
4. Detect whether two strings are one edit away.
5. Implement an LRU cache explanation, even if you do not code the full class.

In [ ]:
def merge_sorted(left, right):
    merged = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

print(merge_sorted([1, 4, 9], [2, 3, 10]))

## 15. Rapid-Fire Theory Checks

Answer these out loud before your interview:

- What is the difference between a shallow copy and a deep copy?
- Why is `list.append` amortized $O(1)$?
- When would you use a generator instead of returning a list?
- What problem does a context manager solve?
- What is the difference between `is` and `==`?
- Why can mutable default arguments be dangerous?
- What is hashability, and why does it matter for dictionaries and sets?

## 16. Senior-Level Framing Tips

In your answers, aim to sound like someone who writes production Python:

- State the straightforward solution first.
- Mention complexity only when it matters.
- Call out edge cases early.
- Prefer readable names and small functions.
- Mention testing strategy for non-trivial logic.
- If multiple designs work, explain the tradeoff instead of pretending there is one perfect answer.

## 17. Final Revision Checklist

Before the interview, make sure you can confidently:

- write list, dict, and set comprehensions
- explain mutability and object references
- use `Counter`, `defaultdict`, and `deque` naturally
- write a clean generator
- explain decorators and closures at a practical level
- model simple data with `@dataclass`
- handle errors without hiding failures
- add concise type hints to function signatures